<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab3_Function_Calling.ipynb)

![Module 3 Lab 3 - Function Calling](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M03/assets/images/M03_Lab3_Function_Calling_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 1 — Function Calling Techniques')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

# From Text to Action: Function Calling

Until now the model only produced **text**. Function calling lets it **take action**: you describe functions ("tools"), the model decides which to call and with what arguments, you run them, and hand the results back. This is the exact mechanism that turns a chatbot into an **agent**.

We'll build up to a small assistant that answers real questions by calling real tools.

## 1. The problem: models are bad at exact work

LLMs predict text; they do not compute. Ask for precise arithmetic and you are trusting a guess. Let's see.

In [ ]:
# ==========================================================
# 1. The model on its own: can it do exact math?
# ==========================================================
hard = "What is 48239 * 7854 + 991? Reply with just the number."
r = client.chat.completions.create(model=DEFAULT_MINI_MODEL,
                                   messages=[{"role": "user", "content": hard}])
model_says = r.choices[0].message.content.strip()
truth = 48239 * 7854 + 991                       # the real answer, computed in Python

pp({"question": hard, "model said": model_says, "correct answer": truth,
    "did it match?": str(truth) in model_says},
   title="Trusting the model for math is risky")

## 2. Give it a calculator tool

A **tool** is a function you describe to the model with a JSON schema (name, purpose, arguments). The model reads it and decides when to use it. Here is a real, safe calculator.

In [ ]:
# ==========================================================
# 2. Define a real calculator tool (safe arithmetic, no eval risk)
# ==========================================================
import ast, operator, json

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg}

def _ev(node):                                   # walk the math expression safely
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp):    return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp):  return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("unsupported expression")

def calculate(expression):
    """Evaluate a plain arithmetic expression like '48239 * 7854 + 991'."""
    return _ev(ast.parse(expression, mode="eval").body)

calc_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression exactly.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string",
                           "description": "e.g. '48239 * 7854 + 991'"}},
            "required": ["expression"],
        },
    },
}
print("calculate('48239 * 7854 + 991') =", calculate("48239 * 7854 + 991"))

### The model calls the tool, we run it, it answers

Same question, but now the model can call `calculate`. Watch it hand off the arithmetic and come back with an exact answer.

In [ ]:
# ==========================================================
# 3. The call -> execute -> return loop (one tool)
# ==========================================================
messages = [{"role": "user", "content": "What is 48239 * 7854 + 991?"}]
resp = client.chat.completions.create(model=DEFAULT_MINI_MODEL, messages=messages,
                                      tools=[calc_tool], tool_choice="auto")
msg = resp.choices[0].message
tc = msg.tool_calls[0]
args = json.loads(tc.function.arguments)         # the model filled in the expression
pp({"model wants to call": tc.function.name, "arguments": args}, title="Tool call")

result = calculate(**args)                        # WE run the function
messages.append(msg)
messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
final = client.chat.completions.create(model=DEFAULT_MINI_MODEL, messages=messages)
pretty_print(final.choices[0].message.content, title="Exact answer, via the tool")

## 3. A real live-data tool

Tools shine most when they fetch things the model cannot know, like **live prices**. This one calls CoinGecko for a real crypto price.

In [ ]:
# ==========================================================
# 4. A real tool: live crypto price (with polite rate-limit retry)
# ==========================================================
import requests, time

def cg_get(url, params, tries=4, timeout=20):
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout)
        if r.status_code == 429:                  # rate limited -> wait and retry
            time.sleep(2 ** i); continue
        r.raise_for_status(); return r
    r.raise_for_status(); return r

def get_crypto_price(coin):
    """Live USD price and 24h change for a coin id, e.g. 'bitcoin'."""
    d = cg_get("https://api.coingecko.com/api/v3/simple/price",
               {"ids": coin, "vs_currencies": "usd", "include_24hr_change": "true"}).json()[coin]
    return {"coin": coin, "price_usd": d["usd"], "change_24h_pct": round(d.get("usd_24h_change", 0), 2)}

price_tool = {
    "type": "function",
    "function": {
        "name": "get_crypto_price",
        "description": "Get the current USD price and 24h change for a crypto coin.",
        "parameters": {
            "type": "object",
            "properties": {"coin": {"type": "string", "description": "coin id, e.g. bitcoin, ethereum, solana"}},
            "required": ["coin"],
        },
    },
}
pp(get_crypto_price("bitcoin"), title="Live price (direct call)")

## 4. Chaining tools: a mini agent

The real magic: give the model **all** the tools and let it loop, calling one tool, seeing the result, then calling another until it can answer. That loop is the heart of an AI **agent**.

In [ ]:
# ==========================================================
# 5. The agent loop: many tools, call until the model is done
# ==========================================================
TOOLS = [calc_tool, price_tool]
FUNCTIONS = {"calculate": calculate, "get_crypto_price": get_crypto_price}

def run_assistant(question, max_steps=6, show=True):
    """Give the model every tool and let it call them in a loop until it answers."""
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = client.chat.completions.create(model=DEFAULT_MINI_MODEL, messages=messages,
                                              tools=TOOLS, tool_choice="auto")
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:                    # no more tools -> final answer
            return msg.content
        for tc in msg.tool_calls:                 # run every tool the model asked for
            args = json.loads(tc.function.arguments)
            out = FUNCTIONS[tc.function.name](**args)
            if show: print(f"  -> {tc.function.name}({args}) = {out}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)})
    return "(stopped: too many steps)"

print("Steps the assistant took:")
answer = run_assistant("What is 15% of the current price of Bitcoin?")
pretty_print(answer, title="Ask anything: a two-tool answer")

> **Pause and think.** For that last question the model had to (1) fetch a live price, then (2) do exact math on it, then answer, all from one English sentence. That call-see-decide loop is exactly how AI agents work. Where in your job is a question really "look something up, then compute, then answer"?

**Your notes** *(double-click to edit)*

- A real question that needs look-up + compute: 
- Which tools it would need: 

## 5. Hands-on: add your own tool

Write a new function, describe it as a tool, add both to `FUNCTIONS` and `TOOLS`, then ask the assistant a question that needs it (for example a `convert_currency` tool, or `days_between(date1, date2)`).

In [ ]:
# ==========================================================
# 6. Hands-on: add a tool to the assistant (fill in the -----)
# ==========================================================
def my_function(-----):
    """Describe what your tool does."""
    return -----

my_tool = {
    "type": "function",
    "function": {
        "name": "-----",
        "description": "-----",
        "parameters": {
            "type": "object",
            "properties": {"-----": {"type": "-----", "description": "-----"}},
            "required": ["-----"],
        },
    },
}

# Register it with the assistant, then ask a question that should trigger it
FUNCTIONS["-----"] = my_function
TOOLS.append(my_tool)
pretty_print(run_assistant("-----"), title="Your tool in action")

## Wrap-up

You closed the loop from text to action: **define tools -> the model requests a call -> you execute -> you return the result -> the model answers**, and then let it **loop** across tools to solve a multi-step question. Combined with JSON mode and Pydantic (Lab 2), you now control the whole pipeline. That call-see-decide loop is the foundation of the **AI agents** you build next module.